# Demo - Prompting Techniques on a Local Model
**Day 2 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-prompting-techniques.ipynb)

**Goal:** Show zero-shot, few-shot, and chain-of-thought prompting without any
API key, using a small instruction-tuned model that runs on the Colab CPU.

We use `google/flan-t5-base`, an instruction-following model, loaded directly
with transformers. The same techniques transfer directly to GPT, Claude, and
Gemini; only the client call changes. Decoding is greedy so the demo is
repeatable.

No API key needed. The model downloads once (~1 GB), then caches. For a
smaller download set `MODEL_NAME` to `"google/flan-t5-small"`.

In [1]:
# Scoped install for this demo (safe to re-run; often preinstalled on Colab).
%pip install -q torch transformers sentencepiece

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/flan-t5-base"

In [3]:
print(f"Loading {MODEL_NAME} (downloads once, then cached)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()


def make_generator(model, tokenizer):
    """Return a greedy text-in, text-out function backed by flan-t5."""
    def generate(prompt, max_new_tokens=64):
        inputs = tokenizer(prompt, return_tensors="pt")
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
        return tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generate


def run(generate, label, prompt):
    """Print a labeled prompt and the model's answer."""
    answer = generate(prompt).strip()
    print(f"\n{label}")
    print(f"  prompt : {prompt}")
    print(f"  answer : {answer}")


generate = make_generator(model, tokenizer)

Loading google/flan-t5-base (downloads once, then cached)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 1. Zero-shot
No examples, rely on what the model already knows.

In [4]:
def demo_zero_shot(generate):
    print("1. Zero-shot (no examples, rely on what the model knows)")
    run(
        generate,
        "Sentiment classification",
        "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. "
        "Text: The update completely broke my workflow.",
    )


demo_zero_shot(generate)

1. Zero-shot (no examples, rely on what the model knows)

Sentiment classification
  prompt : Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Text: The update completely broke my workflow.
  answer : NEGATIVE


## 2. Few-shot
A few examples teach the pattern and format.

In [5]:
def demo_few_shot(generate):
    print("2. Few-shot (a few examples teach the pattern and format)")
    run(
        generate,
        "Sentiment with examples",
        "Classify the sentiment.\n"
        "Text: Best purchase ever. -> POSITIVE\n"
        "Text: Broke after one day. -> NEGATIVE\n"
        "Text: The battery could be better but I like it. ->",
    )


demo_few_shot(generate)

2. Few-shot (a few examples teach the pattern and format)

Sentiment with examples
  prompt : Classify the sentiment.
Text: Best purchase ever. -> POSITIVE
Text: Broke after one day. -> NEGATIVE
Text: The battery could be better but I like it. ->
  answer : positive


## 3. Chain-of-thought
Ask for step-by-step reasoning to fix a wrong direct answer on multi-step
math.

In [6]:
def demo_chain_of_thought(generate):
    print("3. Chain-of-thought (ask for step-by-step reasoning)")
    run(
        generate,
        "Direct answer (small model slips on multi-step math)",
        "A week has 7 days. How many days are in 3 weeks?",
    )
    run(
        generate,
        "With 'think step by step'",
        "A week has 7 days. How many days are in 3 weeks? Think step by step.",
    )


demo_chain_of_thought(generate)

3. Chain-of-thought (ask for step-by-step reasoning)

Direct answer (small model slips on multi-step math)
  prompt : A week has 7 days. How many days are in 3 weeks?
  answer : 7 days / week * 7 days / week = 96 days. 3 weeks / week * 96 days / week = 96 days.

With 'think step by step'
  prompt : A week has 7 days. How many days are in 3 weeks? Think step by step.
  answer : A week has 7 days so in 3 weeks there are 7 * 3 = 21 days. So, the answer is 21.


## Expected output
- Zero-shot returns a sentiment label, usually NEGATIVE, for the broken
  workflow text.
- Few-shot follows the `->` format shown in the examples and returns a label
  for the mixed-sentiment battery text.
- Chain-of-thought: the direct answer may be wrong or terse, while adding
  "Think step by step" often nudges the small model toward the correct answer
  (21 days) by walking through the arithmetic.

**Takeaway:** the technique lives in the prompt, not the provider. The same
zero/few-shot and CoT patterns work on any LLM API.